# Wine Quality Analysis — White Wine (Classification & Regression)

## Objective

Predict red wine quality using both classification and regression approaches based on physicochemical properties.

- **Classification**: Predict quality categories (Low / Medium / High)
- **Regression**: Predict the original quality score (observed range: 3–8)

## Design Decisions

- **Label regrouping for classification**:
  - Low: quality ≤ 5
  - Medium: quality = 6
  - High: quality ≥ 7

- **Separate analysis**:
  - Red and white wines are analysed separately because their physicochemical distributions differ.

- **Six models are evaluated**:
  - Classification: GaussianNB, Logistic Regression, Random Forest Classifier
  - Regression: Linear Regression, Random Forest Regressor, Gradient Boosting Regressor

- **Evaluation metrics**:
  - Classification: macro-F1 and accuracy
  - Regression: RMSE, MAE, and R²

## Improvements Implemented

- Data exploration (EDA) and feature analysis
- Missing value and duplicate checks, with duplicates removed
- Correlation analysis
- Outlier-robust scaling using `RobustScaler` for Logistic Regression
- Feature scaling using `StandardScaler` for Linear Regression
- Class weighting to handle classification imbalance
- Stratified 5-fold cross-validation for classification
- 5-fold cross-validation for regression
- `GridSearchCV` hyperparameter tuning
- Overfitting analysis using train vs test performance
- Error analysis using confusion matrices and residual plots


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, KFold
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    accuracy_score
)

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

## Load White Wine Dataset

We unzip the dataset and load the white wine CSV file.
White wine is processed in its own notebook to ensure separate analysis and reporting.

In [ ]:
!unzip wine+quality.zip -d wine+quality

unzip:  cannot find or open wine+quality.zip, wine+quality.zip.zip or wine+quality.zip.ZIP.


In [ ]:
WHITE_PATH = "wine+quality/winequality-white.csv"
df = pd.read_csv(WHITE_PATH, sep=";")

print("White wine dataset shape:", df.shape)
display(df.head())
display(df.describe())

FileNotFoundError: [Errno 2] No such file or directory: 'wine+quality/winequality-white.csv'

## Data Quality Checks (EDA)

We check:
- Missing values
- Duplicate rows

Duplicates (if any) are removed to prevent repeated samples affecting training/evaluation.

In [ ]:
print("Missing values per column:")
print(df.isna().sum())

dup_count = df.duplicated().sum()
print("\nDuplicate rows BEFORE removal:", dup_count)

if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)

print("Shape AFTER duplicate removal:", df.shape)
print("Duplicate rows AFTER removal:", df.duplicated().sum())

## Label Transformation (Regrouping)

The original integer quality score is regrouped into three classes:
- Low (≤5), Medium (=6), High (≥7)

Rationale:
- Improves interpretability and stabilises classification.
- Reduces sensitivity to small rating differences in subjective quality scores.

In [ ]:
def regroup_quality(q: int) -> str:
    if q <= 5:
        return "Low"
    elif q == 6:
        return "Medium"
    else:
        return "High"

df["quality_label"] = df["quality"].apply(regroup_quality)

print("Original quality distribution:")
print(df["quality"].value_counts().sort_index())

print("\nRegrouped label distribution:")
print(df["quality_label"].value_counts())


## EDA: Class Distribution (Imbalance)

White wine typically exhibits a different imbalance pattern from red wine.
The Medium-quality class often dominates, while High-quality remains a minority.

This motivates:
- **macro-F1** as the primary metric (class-balanced evaluation)
- **class_weight** in training to reduce majority-class bias

Class imbalance is checked for the classification target because Low, Medium, and High labels may not be equally represented. For regression, the original quality score is kept as a continuous/ordinal target.

In [ ]:
counts = df["quality_label"].value_counts()
pct = df["quality_label"].value_counts(normalize=True) * 100

display(pd.DataFrame({"count": counts, "percent(%)": pct.round(2)}))

plt.figure()
plt.bar(counts.index, counts.values)
plt.title("Class Distribution (White Wine)")
plt.ylabel("Count")
plt.show()

## EDA: Feature Distributions (Outlier Evidence)

We plot feature distributions to check skewness and potential extreme values.
Because several features can be skewed, RobustScaler is used for scale-sensitive models
(e.g., Logistic Regression) since it is less affected by outliers than StandardScaler.


In [ ]:
df.drop(columns=["quality_label"]).hist(bins=20, figsize=(14, 10))
plt.suptitle("Feature Distributions (White Wine)")
plt.show()

## Feature Analysis: Correlation

We separately examine:
1) **Feature–feature correlation** (multicollinearity evidence)
2) **Feature–quality correlation** (interpretability)

This is important for explaining why some models (e.g., Random Forest)
may capture complex interactions better than linear models.
Note: correlation indicates association, not causation.


In [ ]:
# Prepare feature matrix X (exclude target columns) for feature-feature analysis
X_corr = df.drop(columns=["quality", "quality_label"])

# (1) Feature-feature correlation (absolute)
corr_x = X_corr.corr(numeric_only=True).abs()
np.fill_diagonal(corr_x.values, 0)

top_pairs = (
    corr_x.unstack()
          .sort_values(ascending=False)
          .drop_duplicates()
          .head(10)
)

print("Top 10 feature-feature correlations (absolute):")
print(top_pairs)

# (2) Feature-quality correlation
corr_all = df.drop(columns=["quality_label"]).corr(numeric_only=True)
quality_corr = corr_all["quality"].sort_values(ascending=False)

print("\nTop positive correlations with quality:")
print(quality_corr.head(6))

print("\nTop negative correlations with quality:")
print(quality_corr.tail(6))

## Train/Test Split

We keep 20% of the dataset as a held-out test set for final evaluation only.  
The same feature split is shared by both classification and regression models so that all six models are evaluated on the same training and test samples.

The training set is used for:

- cross-validation benchmarking
- GridSearchCV hyperparameter tuning
- model fitting

For classification, stratification is applied using the regrouped quality labels to preserve the Low / Medium / High class proportions in both training and test sets.

In [ ]:
# Features
X = df.drop(columns=["quality", "quality_label"])

# Targets
y_cls = df["quality_label"]   # Classification target: Low / Medium / High
y_reg = df["quality"]         # Regression target: original quality score

# Shared train/test split for both classification and regression
X_train, X_test, y_train_cls, y_test_cls, y_train_reg, y_test_reg = train_test_split(
    X, y_cls, y_reg,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)

print("Classification target:")
print("y_train_cls shape:", y_train_cls.shape)
print("y_test_cls shape: ", y_test_cls.shape)

print("\nRegression target:")
print("y_train_reg shape:", y_train_reg.shape)
print("y_test_reg shape: ", y_test_reg.shape)

## Classification Models (3 Models as Required)

- GaussianNB: probabilistic baseline
- Logistic Regression: linear baseline (RobustScaler + class_weight)
- Random Forest: nonlinear ensemble (class_weight)

This comparison tests whether relationships between chemical properties and quality are linear or nonlinear.

In [ ]:
models = {
    "GaussianNB": GaussianNB(),

    "LogisticRegression": Pipeline([
        ("scaler", RobustScaler()),
        ("clf", LogisticRegression(
            max_iter=4000,
            solver="lbfgs",
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=42
    )
}

## Stratified 5-Fold Cross-Validation

Cross-validation reduces dependence on a single train–test split
and provides a more reliable estimate of model generalisation performance.

Primary metric: macro-F1  
Secondary metric: accuracy

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "acc": "accuracy",
    "f1_macro": "f1_macro",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro"
}

rows = []
for name, model in models.items():
    out = cross_validate(model, X_train, y_train_cls, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({
        "Model": name,
        "CV_Acc_mean": out["test_acc"].mean(),
        "CV_Acc_std": out["test_acc"].std(),
        "CV_F1macro_mean": out["test_f1_macro"].mean(),
        "CV_F1macro_std": out["test_f1_macro"].std(),
        "CV_Prec_macro_mean": out["test_precision_macro"].mean(),
        "CV_Rec_macro_mean": out["test_recall_macro"].mean(),
    })

cv_table = pd.DataFrame(rows).sort_values("CV_F1macro_mean", ascending=False)
display(cv_table)

## GridSearchCV (Random Forest) — Computation-Aware Tuning for White Wine

The white wine dataset is larger than the red wine dataset, so an exhaustive grid with 5-fold CV
can be computationally expensive.

To reduce runtime while keeping tuning scientifically valid:
- We use **Stratified 3-fold CV** for hyperparameter search.
- We search a **compact grid** focused on model complexity controls (e.g., `max_depth`, `min_samples_leaf`),
  which are most relevant for reducing overfitting.
- During tuning we use a **moderate number of trees** to speed up evaluation.
  After selecting the best hyperparameters, we **retrain the final model with more trees**
  to improve stability.

The optimisation objective remains **macro-F1** to account for class imbalance.

In [ ]:
rf = RandomForestClassifier(
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

# Compact grid (focus on complexity/overfitting control)
param_grid_fast = {
    "n_estimators": [150, 250],     # smaller during tuning
    "max_depth": [10, 20, None],
    "min_samples_leaf": [2, 4],     # key regularisation (skip 1 for stability)
    "max_features": ["sqrt"]        # fixed to reduce search size
}

cv_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid = GridSearchCV(
    rf,
    param_grid=param_grid_fast,
    scoring="f1_macro",
    cv=cv_tune,         # 3-fold for speed
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train_cls)

best_params_final = grid.best_params_.copy()
best_params_final.pop("n_estimators", None)

best_rf_clf = RandomForestClassifier(
    **best_params_final,
    n_estimators=500,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

best_rf_clf.fit(X_train, y_train_cls)

## Overfitting Analysis

We compare:
- Training vs test macro-F1
- CV train vs CV validation macro-F1 (more reliable generalisation signal)
- Best CV macro-F1 vs test macro-F1

GridSearchCV for the white wine dataset uses a **computation-aware 3-fold stratified CV**
to reduce runtime due to the larger dataset size.

For overfitting diagnosis, we additionally evaluate the selected model using a
**5-fold cross-validation train–validation comparison**, which provides a more stable
estimate of generalisation performance.

A large gap indicates overfitting; a small gap suggests better generalisation.

In [ ]:
train_pred_cls = best_rf_clf.predict(X_train)
test_pred_cls  = best_rf_clf.predict(X_test)

train_f1 = f1_score(y_train_cls, train_pred_cls, average="macro")
test_f1  = f1_score(y_test_cls,  test_pred_cls,  average="macro")

train_acc = accuracy_score(y_train_cls, train_pred_cls)
test_acc  = accuracy_score(y_test_cls,  test_pred_cls)

print("Train macro-F1:", round(train_f1, 4), " Train Acc:", round(train_acc, 4))
print("Test  macro-F1:", round(test_f1, 4), " Test  Acc:", round(test_acc, 4))
print("Best CV macro-F1 (GridSearch):", round(grid.best_score_, 4))

gap = train_f1 - test_f1
print("Macro-F1 gap (train - test):", round(gap, 4))

# CV train vs valid (more reliable)
cv_overfit = cross_validate(
    best_rf_clf, X_train, y_train_cls,
    cv=cv,
    scoring="f1_macro",
    return_train_score=True,
    n_jobs=-1
)

print("CV train macro-F1 mean:", round(cv_overfit["train_score"].mean(), 4))
print("CV valid macro-F1 mean:", round(cv_overfit["test_score"].mean(), 4))
print("CV gap (train - valid):", round((cv_overfit["train_score"].mean() - cv_overfit["test_score"].mean()), 4))

## Final Evaluation & Error Analysis (Held-out Test)

We report:
- classification report (per-class precision/recall/F1)
- confusion matrix (raw and normalized)
- top confusion pairs (True → Pred)

For white wine, errors often concentrate around the dominant Medium class,
and misclassifications frequently occur between adjacent classes (Medium ↔ High or Low ↔ Medium).

In [ ]:
print("=== Classification Report (Test) ===")
print(classification_report(y_test_cls, test_pred_cls, digits=4))

labels = ["Low", "Medium", "High"]
cm = confusion_matrix(y_test_cls, test_pred_cls, labels=labels)

ConfusionMatrixDisplay(cm, display_labels=labels).plot(values_format="d")
plt.title("Confusion Matrix (Raw) — White Wine")
plt.show()

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm, display_labels=labels).plot(values_format=".2f")
plt.title("Confusion Matrix (Normalized) — White Wine")
plt.show()

pairs = []
for i, t in enumerate(labels):
    for j, p in enumerate(labels):
        if i != j:
            pairs.append((t, p, cm[i, j]))
pairs.sort(key=lambda x: x[2], reverse=True)

print("Top confusions (True -> Pred):")
for a, b, c in pairs[:3]:
    print(f"{a} -> {b}: {c}")

## Test Set Comparison

We verify stratification by checking the label distribution in the test set.
This supports fair interpretation of per-class metrics.

In [ ]:
test_counts = y_test_cls.value_counts()
test_pct = y_test_cls.value_counts(normalize=True) * 100
display(pd.DataFrame({"test_count": test_counts, "test_percent(%)": test_pct.round(2)}))

## Feature Importance (Random Forest)

Random Forest feature importance supports interpretability.
For white wine, features such as density and residual sugar are often influential,
reflecting different production characteristics compared with red wine.

In [ ]:
importances = best_rf_clf.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

display(feat_imp.head(10))

plt.figure(figsize=(8, 5))
plt.barh(feat_imp.index[:10][::-1], feat_imp.values[:10][::-1])
plt.title("Top 10 Feature Importances (RF) — White Wine")
plt.xlabel("Importance")
plt.show()

## Regression Models (3 Models as Required)

- **Linear Regression**: baseline regression model to capture linear relationships between features and wine quality.
- **Random Forest Regressor**: nonlinear ensemble model that captures complex feature interactions and reduces overfitting through averaging.
- **Gradient Boosting Regressor**: advanced boosting model that sequentially improves predictions and captures complex nonlinear patterns.

This comparison tests whether relationships are primarily linear or nonlinear.

## Linear Regression

Linear Regression is the baseline model. It models wine quality as a weighted sum of features:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \cdots + w_{11} x_{11}$$

The model minimises Mean Squared Error (MSE) via Ordinary Least Squares (OLS).
StandardScaler is used inside a Pipeline since Linear Regression is sensitive to feature scale.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

reg_cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# Pipeline: StandardScaler + LinearRegression
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("linreg", LinearRegression())
])

param_grid_lr = {
    "linreg__fit_intercept": [True, False]
}

lr_grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid_lr,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

lr_grid.fit(X_train, y_train_reg)

print("Best Parameters:", lr_grid.best_params_)
print("Best CV RMSE:", round(np.sqrt(-lr_grid.best_score_), 4))

In [ ]:
# Prediction
best_lr_reg = lr_grid.best_estimator_
y_pred_lr = best_lr_reg.predict(X_test)

In [ ]:
# Evaluation
lr_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_lr))
lr_mae  = mean_absolute_error(y_test_reg, y_pred_lr)
lr_r2   = r2_score(y_test_reg, y_pred_lr)

print(f"RMSE : {lr_rmse:.4f}")
print(f"MAE  : {lr_mae:.4f}")
print(f"R2   : {lr_r2:.4f}")

In [ ]:
# Overfitting check
y_train_pred_lr = best_lr_reg.predict(X_train)
y_test_pred_lr  = best_lr_reg.predict(X_test)

train_rmse_lr = np.sqrt(mean_squared_error(y_train_reg, y_train_pred_lr))
test_rmse_lr  = np.sqrt(mean_squared_error(y_test_reg,  y_test_pred_lr))
train_r2_lr   = r2_score(y_train_reg, y_train_pred_lr)
test_r2_lr    = r2_score(y_test_reg,  y_test_pred_lr)

print(f"Train RMSE: {train_rmse_lr:.4f}  |  Test RMSE: {test_rmse_lr:.4f}")
print(f"Train R2  : {train_r2_lr:.4f}  |  Test R2  : {test_r2_lr:.4f}")

## Residual Plot - Linear Regression
For Linear Regression, the residual plot is especially useful for checking whether the linearity assumption is reasonable and whether the model suffers from heteroscedasticity or underfitting.

In [ ]:
# Residual plot

residuals = y_test_reg - y_test_pred_lr

plt.figure(figsize=(8, 4))
plt.scatter(y_test_pred_lr, residuals, alpha=0.4)
plt.axhline(0, color="red", linewidth=1)
plt.xlabel("Predicted Quality")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot — Linear Regression (White Wine)")
plt.tight_layout()
plt.show()

For Linear Regression, the residuals are more widely dispersed around the zero line, indicating that prediction errors are relatively larger and less stable. This suggests that the model is unable to fully capture the complex relationships between physicochemical features and wine quality using a simple linear relationship.


## Feature coefficients - Linear Regression
For Linear Regression, feature interpretation is performed using regression coefficients because the model assumes a direct linear relationship between the input variables and the target variable. Each coefficient represents the expected change in predicted wine quality for a one-unit increase in the corresponding feature, while holding other variables constant.

In [ ]:
# Feature coefficients

lr_step = best_lr_reg.named_steps["linreg"]
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": lr_step.coef_
}).sort_values("coefficient", ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(coef_df["feature"], coef_df["coefficient"])
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Linear Regression Coefficients (White Wine)")
plt.xlabel("Coefficient Value")
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))

## Random Forest Regressor

Random Forest Regressor is a nonlinear ensemble model used to predict the original white wine quality score. It builds multiple decision trees and averages their predictions, which helps capture complex relationships between physicochemical properties while reducing variance compared with a single decision tree.

Unlike Linear Regression, Random Forest does not require feature scaling because it is tree-based. `GridSearchCV` is used to tune important hyperparameters such as the number of trees, tree depth, and minimum samples per leaf. The model is evaluated using RMSE, MAE, and R², and train-test performance is compared to assess overfitting.

In [ ]:
rf_reg = RandomForestRegressor(random_state=42, n_jobs=-1)

# Hyperparameter tuning grid
rf_reg_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.6, 1.0]
}

rf_reg_grid = GridSearchCV(
    estimator=rf_reg,
    param_grid=rf_reg_param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

rf_reg_grid.fit(X_train, y_train_reg)

print("Best Parameters:", rf_reg_grid.best_params_)
print("Best CV RMSE:", round(np.sqrt(-rf_reg_grid.best_score_), 4))

In [ ]:
# Prediction
best_rf_reg = rf_reg_grid.best_estimator_
y_pred_rf_reg = best_rf_reg.predict(X_test)

# Evaluation
rf_reg_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_rf_reg))
rf_reg_mae = mean_absolute_error(y_test_reg, y_pred_rf_reg)
rf_reg_r2 = r2_score(y_test_reg, y_pred_rf_reg)

print(f"RMSE : {rf_reg_rmse:.4f}")
print(f"MAE  : {rf_reg_mae:.4f}")
print(f"R2   : {rf_reg_r2:.4f}")

In [ ]:
# Overfitting check
y_train_pred_rf_reg = best_rf_reg.predict(X_train)
y_test_pred_rf_reg = best_rf_reg.predict(X_test)

train_rmse_rf_reg = np.sqrt(mean_squared_error(y_train_reg, y_train_pred_rf_reg))
test_rmse_rf_reg = np.sqrt(mean_squared_error(y_test_reg, y_test_pred_rf_reg))

train_r2_rf_reg = r2_score(y_train_reg, y_train_pred_rf_reg)
test_r2_rf_reg = r2_score(y_test_reg, y_test_pred_rf_reg)

print(f"Train RMSE: {train_rmse_rf_reg:.4f}  |  Test RMSE: {test_rmse_rf_reg:.4f}")
print(f"Train R2  : {train_r2_rf_reg:.4f}  |  Test R2  : {test_r2_rf_reg:.4f}")

## Residual Plot - Random Forest Regressor
For Random Forest Regressor, residual plots help determine whether the tree-based ensemble models are producing consistent predictions across different wine quality levels and whether large prediction errors or bias patterns still exist.

In [ ]:
# Residual plot
residuals_rf_reg = y_test_reg - y_test_pred_rf_reg

plt.figure(figsize=(8, 4))
plt.scatter(y_test_pred_rf_reg, residuals_rf_reg, alpha=0.4)
plt.axhline(0, color="red", linewidth=1)
plt.xlabel("Predicted Quality")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot — Random Forest Regressor (White Wine)")
plt.tight_layout()
plt.show()

For Random Forest Regressor, the residuals are generally more concentrated around zero compared to Linear Regression, showing that the model produces more accurate predictions for many wine samples. However, several larger residual outliers are still present, indicating that prediction errors can still vary significantly for certain observations.


## Feature importance - Random Forest Regressor
For Random Forest Regressor, feature importance is used instead because these models are based on ensembles of decision trees rather than linear equations. Feature importance reflects how much each feature contributes to reducing impurity and improving predictive performance throughout the tree-building process.

In [ ]:
# Feature Importance - Random Forest Regressor
rf_reg_importance = pd.Series(
    best_rf_reg.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
rf_reg_importance.plot(kind="bar")
plt.title("Feature Importance — Random Forest Regressor (White Wine)")
plt.xlabel("Features")
plt.ylabel("Importance Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(rf_reg_importance)

## Gradient Boosting
Gradient Boosting Regressor is used as an ensemble boosting regression model. Unlike Linear Regression, it does not assume a linear relationship between physicochemical features and wine quality. Instead, it builds multiple weak decision trees sequentially, where each new tree learns from the errors made by the previous trees. This allows the model to capture complex nonlinear relationships and improve prediction accuracy gradually.

Since Gradient Boosting is tree-based, feature scaling is generally not required. `GridSearchCV` is used to tune important hyperparameters such as the number of estimators, learning rate, and maximum tree depth. The model is evaluated using RMSE, MAE, and R² scores. Train and test performance are also compared to analyse whether the model is overfitting.

In [ ]:
# Set hyper parameters for fine tuning
gb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

gb_grid = GridSearchCV(
    estimator = GradientBoostingRegressor(random_state=42),
    param_grid = gb_param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

gb_grid.fit(X_train, y_train_reg)

print("Best Parameters:", gb_grid.best_params_)

In [ ]:
# use model for prediction
best_model_gb = gb_grid.best_estimator_

y_pred_gb = best_model_gb.predict(X_test)

In [ ]:
# evaluation
mse_gb = mean_squared_error(y_test_reg, y_pred_gb)
rmse_gb = np.sqrt(mse_gb)
mae_gb = mean_absolute_error(y_test_reg, y_pred_gb)
r2_gb = r2_score(y_test_reg, y_pred_gb)

print("RMSE:", rmse_gb)
print("MAE:", mae_gb)
print("R2:", r2_gb)

In [ ]:
# Overfitting checking

# train prediction
y_train_pred_gb = best_model_gb.predict(X_train)

# test prediction
y_test_pred_gb = best_model_gb.predict(X_test)

# Train
train_rmse_gb = np.sqrt(mean_squared_error(y_train_reg, y_train_pred_gb))
train_r2_gb = r2_score(y_train_reg, y_train_pred_gb)

# Test
test_rmse_gb = np.sqrt(mean_squared_error(y_test_reg, y_test_pred_gb))
test_r2_gb = r2_score(y_test_reg, y_test_pred_gb)

print(f"Train RMSE: {train_rmse_gb:.4f}  |  Test RMSE: {test_rmse_gb:.4f}")
print(f"Train R2  : {train_r2_gb:.4f}  |  Test R2  : {test_r2_gb:.4f}")

## Residual Plot - Gradient Boosting Regressor
For Gradient Boosting Regressor, residual plots help determine whether the tree-based ensemble models are producing consistent predictions across different wine quality levels and whether large prediction errors or bias patterns still exist.

In [ ]:
# Residual plot
residuals = y_test_reg - y_test_pred_gb

plt.figure(figsize=(8, 4))
plt.scatter(y_test_pred_gb, residuals, alpha=0.4)
plt.axhline(0, color="red", linewidth=1)
plt.xlabel("Predicted Quality")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot — Gradient Boosting (White Wine)")
plt.tight_layout()
plt.show()

For Gradient Boosting Regressor, the residuals appear more evenly distributed around zero with a relatively consistent spread across different prediction ranges. This suggests that the model achieves more balanced prediction behaviour and better generalisation performance across unseen wine samples.

## Feature importance - Gradient Boosting Regressor
For Gradient Boosting Regressor, feature importance is used instead because these models are based on ensembles of decision trees rather than linear equations. Feature importance reflects how much each feature contributes to reducing impurity and improving predictive performance throughout the tree-building process.

In [ ]:
# Feature Importance - Gradient Boosting

gb_importance = pd.Series(
    best_model_gb.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 5))

gb_importance.plot(kind="bar")

plt.title("Feature Importance - Gradient Boosting Regressor (White wine)")
plt.xlabel("Features")
plt.ylabel("Importance Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.show()

print(gb_importance)

##From Feature Coefficients and Feature Importance of 3 Models
Across all three models, alcohol consistently appears as one of the most influential features related to white wine quality. Free sulfur dioxide and residual sugar also show relatively strong influence in several models, while volatile acidity generally demonstrates negative influence on wine quality prediction. These results suggest that the models identified similar important factors affecting white wine quality.

## Regression Model Performance Comparison
A regression model performance comparison table is created to provide a clearer evaluation of the three regression models. Instead of relying on a single metric, multiple evaluation metrics are compared simultaneously to better understand each model’s predictive accuracy and generalisation performance.

RMSE is used to measure the average magnitude of prediction errors, where lower values indicate better prediction accuracy. R² score measures how well the model explains the variance in wine quality, where higher values indicate better model performance.

Both training and testing metrics are included to analyse whether the models are overfitting. A large gap between training and testing performance may indicate that the model memorises the training data but does not generalise well to unseen data.

By comparing Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor together, it becomes easier to identify which model provides the best balance between prediction accuracy and generalisation ability.

In [ ]:
metrics_df = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest Regressor",
        "Gradient Boosting Regressor"
    ],

    "Train RMSE": [
        train_rmse_lr,
        train_rmse_rf_reg,
        train_rmse_gb
    ],

    "Test RMSE": [
        test_rmse_lr,
        test_rmse_rf_reg,
        test_rmse_gb
    ],

    "Train R²": [
        train_r2_lr,
        train_r2_rf_reg,
        train_r2_gb
    ],

    "Test R²": [
        test_r2_lr,
        test_r2_rf_reg,
        test_r2_gb
    ]
})

# round to 4 decimal places
metrics_df = metrics_df.round(4)

print(metrics_df)

The comparison table shows clear differences in model behaviour and generalisation performance across the three regression models.

Linear Regression achieved the most consistent training and testing performance, with Train RMSE (0.7443) and Test RMSE (0.7505) remaining very close. Similarly, the Train R² (0.3003) and Test R² (0.2947) values are highly consistent, suggesting that the model does not significantly overfit. However, the relatively low R² scores indicate that Linear Regression may underfit the dataset and is unable to fully capture the complex nonlinear relationships between physicochemical features and wine quality.

Random Forest Regressor achieved extremely strong training performance with a very low Train RMSE (0.2564) and very high Train R² (0.9170). However, its Test RMSE increased substantially to 0.7013 and the Test R² dropped sharply to 0.3841. This large gap between training and testing performance indicates significant overfitting, where the model memorises the training data but generalises less effectively to unseen wine samples.

Gradient Boosting Regressor achieved a better balance between predictive accuracy and generalisation. Although its training performance is weaker than Random Forest, the Train RMSE (0.5705) and Test RMSE (0.7109) remain relatively closer compared to Random Forest. The model also achieved competitive Test R² performance (0.3672) while showing less severe overfitting behaviour.

Overall, Random Forest achieved the strongest training accuracy but suffered from clear overfitting. Linear Regression generalised consistently but showed limited predictive capability due to its linear assumptions. Gradient Boosting Regressor provided a more balanced trade-off between model complexity, prediction accuracy, and generalisation performance.

## Discussion (White Wine)

The white wine quality prediction task was evaluated using both classification and regression approaches. For classification, the original quality scores were regrouped into Low, Medium, and High categories. The class distribution is usually dominated by the Medium category, so macro-F1 is preferred over accuracy because it gives equal importance to all classes. Class weighting is also used to reduce the influence of class imbalance during model training.

For Logistic Regression, `RobustScaler` was used because several physicochemical features show skewness and potential outliers. For Linear Regression, `StandardScaler` was used because the model is sensitive to feature scale and coefficient magnitude.

Tree-based models such as Random Forest and Gradient Boosting are expected to perform strongly because they can capture nonlinear relationships and feature interactions among chemical properties. Linear Regression is kept as a baseline to test whether wine quality can be reasonably explained by linear feature effects.

For classification, most errors are expected to occur between neighbouring quality groups, especially Medium and High or Low and Medium, because their chemical profiles can overlap and wine quality scoring is partly subjective. For regression, residual plots are used to check whether prediction errors are randomly distributed or show systematic bias.

Overfitting is assessed by comparing training and test performance. For classification, this includes train-test macro-F1 gaps and cross-validation train-validation gaps. For regression, train-test RMSE and R² are compared across models. Regularisation-related parameters such as `min_samples_leaf`, `max_depth`, and learning rate are tuned to improve generalisation.